In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# TODO:
1. Wikipedia harvest - Run as is for now, try to understand why it crushes near the end. **---------> DONE!**
2. Extract tables:
    - length between 3 to 10 **---------> DONE!**
    - table is square and parsed normally (no merged cells) **---------> DONE!**
    - full table **---------> DONE!**
    - remove some columns with no relevant data (empty or by keywords) **---------> DONE!**
3. Add to table extraction text titles and relevant paragraphs for the context. **---------> DONE!**
4. Cluster titles to remove similar options + take only one table per page. **---------> DONE!**
5. Create groups - all the table, try to see what is the main entity - return the column name, if they don't have one, discard the table and return 'None'. **---------> DONE!**
6. Create Questions
    - strategy injection, sample a subset of strategies in a shuffled order, if not possible than discard table.
    - make a function to extract the answer per question strategy --> create GT answer.
    - Add the 'group size' strategy.
    - constract the data samples.

### Havest Wikipedia pages with tables:

In [91]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester()
# Run Phase 1 - Harvesting titles based on keywords
harvester.run_complete_harvest()
# Run Phase 2 - Filtering pages for tables
await harvester.filter_pages_async()

Harvesting Pages By Keywords:   0%|          | 0/233 [00:00<?, ?keyword/s]


TOTAL UNIQUE TITLES COLLECTED: 167913



Method,Keyword,Count
Prefix,List of cities,1537
Prefix,List of countries,1278
Prefix,List of towns,501
Prefix,List of villages,483
Prefix,List of districts,375
Prefix,List of municipalities,1021
Prefix,List of communes,194
Prefix,List of counties,259
Prefix,List of provinces,84
Prefix,List of territories,22


Filtering Pages with Tables:   0%|          | 0/167913 [00:00<?, ?page/s]


TOTAL PAGES WITH TABLES COLLECTED: 62306



In [93]:
save_json("table_titles", harvester.valid_titles)

In [3]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

### Parse and filter Wikipedia tables:

In [4]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(table_titles)

Pre-processing: cluster similar pages and discard similarities ---> 6426 pages discarded.


Extracting Tables:   0%|          | 0/55880 [00:00<?, ?Page/s]

Starting Extraction: 55880 titles | Async Limit: 20 | CPU Workers: 19
TOTAL PAGES WITH VALID TABLES: 37946 OUT OF 55880 (67.9%)


In [5]:
validated_tables = load_jsonl(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))

In [35]:
from knowledge_maze.create_dataset.create_questions.table_processing import TablePostProcessor

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
cleaned_data = processor.process()

Filtering & Cleaning:   0%|          | 0/126924 [00:00<?, ?it/s]

Selecting Best Tables:   0%|          | 0/21005 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      126924
Dropped (Single Col):      971
Dropped (Bad Headers):     12972
Dropped (Cell too long):   3492
Short tables found:        54026
Unique source pages:       21005
Final sampled (1/page):    21005
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

,Year,Team,Event,Result,GP,G,A,Pts
0,2005,Canada,U18,,1,1,2,6
1,2007,Canada,WJC,,0,0,0,8
2,2009,Canada,WC,,1,2,3,2
3,2011,Canada,WC,5th,2,3,5,10
4,Junior totals,Junior totals,Junior totals,Junior totals,1,1,2,14
5,Senior totals,Senior totals,Senior totals,Senior totals,3,5,8,12



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Bristol Old Vic > Bristol Old Vic Theatre School > Peter O'Toole Prize (Award recipients and productions involved in)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Year,Recipients,Productions
0,2012,Emily May Smith and Isaac Stanmore,"Wild Oats , Does My Society Look Big in This? ..."
1,2013,Martin Bassindale and Lindsay Dukes,Great Expectations and The Little Mermaid
2,2014,Jonathan Charles,Juno and the Paycock
3,2014,Bethan Nash and Millie Corser,Swallows and Amazons



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
1982 Kansas City Chiefs season > Regular season > Schedule
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Week,Date,Opponent,Result,Record,Venue,Attendance,Recap
0,1,September 12,at Buffalo Bills,L 9–14,0–1,Rich Stadium,76383,Recap
1,2,September 19,San Diego Chargers,W 19–12,1–1,Arrowhead Stadium,60514,Recap
2,3–10,Players' strike,Players' strike,Players' strike,Players' strike,Players' strike,Players' strike,Players' strike
3,11,November 21,at New Orleans Saints,L 17–27,1–2,Louisiana Superdome,39341,Recap
4,12,November 28,at Los Angeles Rams,L 14–20,1–3,Anaheim Stadium,45793,Recap
5,13,December 5,at Pittsburgh Steelers,L 14–35,1–4,Three Rivers Stadium,52090,Recap
6,14,December 12,Los Angeles Raiders,L 16–21,1–5,Arrowhead Stadium,26307,Recap
7,15,December 19,at Denver Broncos,W 37–16,2–5,Mile High Stadium,74192,Recap
8,16,December 26,San Francisco 49ers,L 13–26,2–6,Arrowhead Stadium,24319,Recap
9,17,January 2,New York Jets,W 37–13,3–6,Arrowhead Stadium,11902,Recap



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Cleveland City Stars > Year-by-year
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Year,Division,League,Reg. season,Playoffs,Open Cup,Avg. attendance
0,2007,3,USL Second Division,2nd,Semifinals,2nd Round,1417
1,2008,3,USL Second Division,3rd,Champions,3rd Round,1634
2,2009,2,USL First Division,11th,Did not qualify,2nd Round,1491


### Create Potential Entity Groups:

In [56]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(cleaned_data, 1000))
await group_constructor.get_groups()

Creating groups based on the Wikipedia tables using LLM...


Usage cost: 0.2383$ (Estimated: 0.24$): 100%|██████████████████████████████████████| 1000/1000 [00:27<00:00, 36.39it/s]



════════════════════════════════════════════
📊 TABLE LLM GROUPING SUMMARY
════════════════════════════════════════════
Total tables scanned:      1000
Total groups made:      903
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE Groups ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
--> Political parties and their election results in the 1906 Liverpool City Council election.
--> Studio albums released by Rodney Atkins.
--> Mayors of Trieste, Italy,

,Year,Number of recipients
0,1970,28
1,1971,41
2,1972,50
3,1973,17
4,1974,21
5,1975,15
6,1976,16
7,1977,17
8,1978,0
9,1979,0



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
--> Mary O'Connor (runner) > Achievements
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Year,Competition,Venue,Position,Notes
0,Representing New Zealand,Representing New Zealand,Representing New Zealand,Representing New Zealand,Representing New Zealand
1,1983,World Championships,"Helsinki, Finland",34th,Marathon
2,1984,Olympic Games,"Los Angeles, United States",27th,Marathon


### Create FanOut Questions out of the Entity Groups:

### Testing:

In [69]:
short, long = 0, 0
for l in [len(sample['rows']) for sample in validated_tables]:
    if l > 12: long += 1
    else: short += 1

print(f"Short: {short}, Long: {long}")
print("="*100)

for i in range(3, 13):
    for t in validated_tables:
        if len(t['rows']) == i:
            print(t['table_name'])
            print("="*100)
            tttt = pd.DataFrame(t['rows'])
            display(tttt)
            print("="*100)
            break

Short: 690, Long: 457
EuroLeague Player Of the Year, Group Stage MVP, and Top 16 Stage MVP > FIBA European League ("EuroLeague") Player of the Year (1993–1996)


,Bronze,Member of the FIBA Hall of Fame .
0,Silver,Member of the Naismith Memorial Basketball Hal...
1,Gold,Member of both the FIBA Hall of Fame and the N...
2,Player (X),Denotes the number of times the player has bee...


2014 African Cross Country Championships > Medallists > Individual


,Event,Gold,Silver,Bronze
0,Senior men's 12 km,Leonard Barsoton ( KEN ),34:26.12,Kipruto Kangogo ( KEN )
1,Senior women's 8 km,Faith Chepngetich Kipyegon ( KEN ),25:33.02,Janet Kisa ( KEN )
2,Junior men's 8 km,Moses Letoyie ( KEN ),22:36.80,Andrew Lorot ( KEN )
3,Junior women's 6 km,Agnes Jebet Tirop ( KEN ),18:50.74,Alemitu Heroye ( ETH )


EuroLeague Player Of the Year, Group Stage MVP, and Top 16 Stage MVP > FIBA European League ("EuroLeague") Player of the Year (1993–1996)


...

,Location,Date (from),Date (to),Notes
0,Aqtobe,11 August 1982,July 2013,Service suspended February 4 until 12 June 2013.
1,Almaty,20 April 1944,,See also Trolleybus transport in Almaty
2,Astana,18 January 1983,30 September 2008,
3,Atyrau,4 September 1996,29 April 1999,
4,Novaya Bukhtarma,1 January 1975,1980,"Trolleybus to cement plant, closed because of ..."
5,Petropavl,25 December 1971,1 June 2014,Closed due to bankruptcy.
6,Karaganda,30 May 1967,20 April 2010,
7,Qostanay,28 December 1989,2005,Closed in 2005; contact network fully dismant...
8,Shymkent,11 January 1969,October 2005,"Some operation reported during 2005, none duri..."
9,Taraz,10 April 1980,December 2013,Service suspended from the December 2013 under...


29th Berlin International Film Festival > Official Sections > Retrospective (The following films were shown in the retrospective dedicated to Rudolph Valentino :)


,English title,Original title,Director(s),Production Country
0,All Night,All Night,Paul Powell,United States
1,Blood and Sand,Blood and Sand,Fred Niblo,United States
2,Eyes of Youth,Eyes of Youth,Albert Parker,United States
3,Monsieur Beaucaire,Monsieur Beaucaire,Sidney Olcott,United States
4,Moran of the Lady Letty,Moran of the Lady Letty,George Melford,United States
5,The Conquering Power,The Conquering Power,Rex Ingram,United States
6,The Eagle,The Eagle,Clarence Brown,United States
7,The Four Horsemen of the Apocalypse,The Four Horsemen of the Apocalypse,Rex Ingram,United States
8,The Sheik,The Sheik,George Melford,United States
9,The Son of the Sheik,The Son of the Sheik,George Fitzmaurice,United States


List of awards and nominations received by Edie Falco > Major associations > Golden Globe Awards


,Year,Category,Nominated work,Result
0,2000,Best Actress – Television Series Drama,The Sopranos,Won
1,2001,Best Actress – Television Series Drama,The Sopranos,Nominated
2,2002,Best Actress – Television Series Drama,The Sopranos,Nominated
3,2003,Best Actress – Television Series Drama,The Sopranos,Won
4,2005,Best Actress – Television Series Drama,The Sopranos,Nominated
5,2007,Best Actress – Television Series Drama,The Sopranos,Nominated
6,2008,Best Actress – Television Series Drama,The Sopranos,Nominated
7,2010,Best Actress – Television Series Musical or Co...,Nurse Jackie,Nominated
8,2011,Best Actress – Television Series Musical or Co...,Nurse Jackie,Nominated
9,2014,Best Actress – Television Series Musical or Co...,Nurse Jackie,Nominated


In [53]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester(delay=0.05)
# Run Phase 1 - Harvesting titles based on keywords
raw_titles = harvester.fetch_titles()
# Run Phase 2 - Filtering pages for tables
# table_titles = harvester.filter_pages_with_tables(raw_titles)

Starting Phase 1: Harvesting titles for 18 keywords...


RUNNING: 100%|███████████████████████████| 18/18 [03:44<00:00, 12.49s/keyword, current_kw=missions, total_pages=316557]


Keyword              | Pages Found
----------------------------------
list of              | 306682    
comparison of        | 1401      
rankings of          | 32        
discography of       | 131       
filmography of       | 191       
demographics of      | 1730      
members              | 2915      
teams                | 44        
countries            | 321       
cities               | 674       
species              | 234       
films                | 282       
awards               | 1177      
companies            | 197       
airlines             | 78        
universities         | 338       
planets              | 80        
missions             | 50        
----------------------------------
TOTAL UNIQUE         | 316557    



In [36]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import get_group
try:
    import nest_asyncio
    nest_asyncio.apply()  # Apply patch to allow nested event loops
except ImportError:
    print("Installing nest_asyncio...")


llm_output = get_group(final_data[0])
llm_output

TableGroup(group_description='Bean-to-bar chocolate manufacturers founded 2010-2015', filter_column='Year founded', filter_type='numerical_range', filter_value=[2010, 2015])

In [37]:
import re
import traceback

def filter_table_rows_dict(llm_data, table_rows):
    try:
        # 1. Extract data using Pydantic V2 standards
        if hasattr(llm_data, 'model_dump'):
            data = llm_data.model_dump()
        elif hasattr(llm_data, 'Output') and hasattr(llm_data.Output, 'model_dump'):
            data = llm_data.Output.model_dump()
        elif hasattr(llm_data, 'dict'): # Fallback for V1
            data = llm_data.dict()
        else:
            data = llm_data

        col_name_query = str(data.get('filter_column', "")).lower().strip()
        f_type = str(data.get('filter_type', "")).lower()
        f_val = data.get('filter_value', [])

        if not col_name_query or not table_rows:
            return []

        # 2. Strict Column Matching
        # We look for the BEST match, not just any match
        target_col = None
        for k in table_rows[0].keys():
            # Clean both for a fair comparison
            clean_k = str(k).lower().strip()
            if col_name_query == clean_k or col_name_query in clean_k:
                target_col = k
                break
        
        if target_col is None:
            return []

        filtered_rows = []
        for row in table_rows:
            val_orig = row.get(target_col)
            if val_orig is None: continue
            
            # Sanitize cell content (Handles tuples/lists from scraper)
            if isinstance(val_orig, (tuple, list)) and len(val_orig) > 0:
                cell_str = str(val_orig[0])
            else:
                cell_str = str(val_orig)
            
            cell_str = cell_str.strip()
            if not cell_str: continue

            # --- Strict Filter Logic ---
            if 'list' in f_type:
                # Exact or substring match for list items
                low_cell = cell_str.lower()
                if any(str(item).lower() in low_cell for item in f_val):
                    filtered_rows.append(row)

            elif 'num' in f_type:
                try:
                    # Extract only the first number found (ignores citations like [1])
                    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", cell_str.replace(',', ''))
                    if not numbers: continue
                    
                    val_as_float = float(numbers[0])
                    low_bound = float(f_val[0])
                    high_bound = float(f_val[1])
                    
                    # Core check
                    if low_bound <= val_as_float <= high_bound:
                        filtered_rows.append(row)
                except:
                    continue

            elif 'temp' in f_type:
                try:
                    # Look for 4-digit years
                    year_match = re.search(r'\b(1\d{3}|20\d{2})\b', cell_str)
                    if year_match:
                        row_year = int(year_match.group(0))
                        # Handle date strings or years in f_val
                        start_y = int(str(f_val[0])[:4])
                        end_y = int(str(f_val[1])[:4])
                        if start_y <= row_year <= end_y:
                            filtered_rows.append(row)
                except:
                    continue

        return filtered_rows

    except Exception as e:
        print(f"CRITICAL FILTER ERROR: {e}")
        traceback.print_exc()
        return []

In [39]:
# After your LLM call
filtered_data = filter_table_rows_dict(llm_output, final_data[0]['rows'])

In [42]:
import re

def extract_entity_list(llm_data, table_rows):
    try:
        # Pydantic V2 migration safe
        data = llm_data.model_dump() if hasattr(llm_data, 'model_dump') else llm_data
        
        ent_col_query = str(data.get('entity_column', "")).lower().strip()
        fil_col_query = str(data.get('filter_column', "")).lower().strip()
        f_type = data.get('filter_type')
        f_val = data.get('filter_value')
        sort_order = data.get('sort_order', 'desc')

        if not table_rows: return []

        # Find the actual column keys
        keys = table_rows[0].keys()
        entity_key = next((k for k in keys if ent_col_query in str(k).lower()), None)
        filter_key = next((k for k in keys if fil_col_query in str(k).lower()), None)

        if not entity_key or not filter_key: return []

        # --- STEP 1: Numerical Cleaning for Sorting/Filtering ---
        processed_data = []
        for row in table_rows:
            raw_ent = row.get(entity_key, "Unknown")
            raw_fil = row.get(filter_key, 0)
            
            # Extract name (handle tuples/lists from scraper)
            name = str(raw_ent[0] if isinstance(raw_ent, (list, tuple)) else raw_ent).strip()
            
            # Extract number for sorting/filtering
            clean_fil = re.sub(r'[^\d.]', '', str(raw_fil))
            val_as_float = float(clean_num[0]) if (clean_num := re.findall(r"[-+]?\d*\.\d+|\d+", str(raw_fil).replace(',', ''))) else 0.0
            
            processed_data.append({"name": name, "val": val_as_float, "original_row": row})

        # --- STEP 2: Apply Logic ---
        result_entities = []

        if f_type == "top_n":
            # Sort by the numerical value
            sorted_data = sorted(processed_data, key=lambda x: x['val'], reverse=(sort_order == 'desc'))
            # Take exactly N
            limit = min(max(int(f_val), 3), 10) # Safety clamp between 3 and 10
            result_entities = [d['name'] for d in sorted_data[:limit]]

        elif f_type == "numerical_range":
            # Traditional slice
            low, high = float(f_val[0]), float(f_val[1])
            matches = [d['name'] for d in processed_data if low <= d['val'] <= high]
            result_entities = matches[:10] # Ensure we don't exceed 10

        return result_entities

    except Exception as e:
        print(f"Extraction Error: {e}")
        return []

In [47]:
final_data[2]

{'table_name': 'List of Spanish films of 2017 > Box office',
 'rows': [{'Rank': 1,
   'Title': 'Tad the Lost Explorer and the Secret of King Midas (Tadeo Jones 2: El secreto del rey Midas)',
   'Distributor': 'Paramount Pictures',
   'Admissions': 3167110,
   'Domestic gross (€)': 17553149},
  {'Rank': 2,
   'Title': 'Perfect Strangers (Perfectos desconocidos)',
   'Distributor': 'Universal Pictures',
   'Admissions': 2020940,
   'Domestic gross (€)': 12969658},
  {'Rank': 3,
   'Title': "It's for Your Own Good (Es por tu bien)",
   'Distributor': 'Buena Vista International',
   'Admissions': 1584578,
   'Domestic gross (€)': 9638993},
  {'Rank': 4,
   'Title': 'Marrowbone (El secreto de Marrowbone)',
   'Distributor': 'Universal Pictures',
   'Admissions': 1158521,
   'Domestic gross (€)': 7153428},
  {'Rank': 5,
   'Title': 'Lord, Give Me Patience (Señor, dame paciencia)',
   'Distributor': 'Warner Bros. Pictures',
   'Admissions': 1106441,
   'Domestic gross (€)': 6682652},
  {'Rank

In [54]:
llm_output = get_group(final_data[0])
llm_output

TableGroup(group_description='Latest 7 chocolate manufacturers by founding date', entity_column='Company', filter_column='Year founded', filter_type='top_n', filter_value=7, sort_order='desc')

In [49]:
extract_entity_list(llm_output, final_data[2]['rows'])

['Tad the Lost Explorer and the Secret of King Midas (Tadeo Jones 2: El secreto del rey Midas)',
 'Perfect Strangers (Perfectos desconocidos)',
 "It's for Your Own Good (Es por tu bien)",
 'Marrowbone (El secreto de Marrowbone)',
 'Lord, Give Me Patience (Señor, dame paciencia)',
 'Toc Toc',
 'The Invisible Guardian (El guardián invisible)']

In [51]:
import re

def extract_grouped_data(llm_data, table_rows):
    try:
        # Pydantic V2 migration safe
        data = llm_data.model_dump() if hasattr(llm_data, 'model_dump') else llm_data
        
        group_desc = data.get('group_description', "")
        ent_col_query = str(data.get('entity_column', "")).lower().strip()
        fil_col_query = str(data.get('filter_column', "")).lower().strip()
        f_type = data.get('filter_type')
        f_val = data.get('filter_value')
        sort_order = data.get('sort_order', 'desc')

        if not table_rows: return {}

        # 1. Identify the actual keys in the original dictionary
        keys = list(table_rows[0].keys())
        entity_key = next((k for k in keys if ent_col_query in str(k).lower()), None)
        filter_key = next((k for k in keys if fil_col_query in str(k).lower()), None)

        if not entity_key or not filter_key: return {}

        # 2. Pre-process and Clean data
        processed_pool = []
        for row in table_rows:
            raw_ent = row.get(entity_key, "Unknown")
            raw_fil = row.get(filter_key, 0)
            
            # Clean entity name
            name = str(raw_ent[0] if isinstance(raw_ent, (list, tuple)) else raw_ent).strip()
            
            # Clean numerical value for sorting/filtering
            # Keep original raw_fil for the "Group entities" value output
            clean_num_list = re.findall(r"[-+]?\d*\.\d+|\d+", str(raw_fil).replace(',', ''))
            val_as_float = float(clean_num_list[0]) if clean_num_list else 0.0
            
            processed_pool.append({
                "name": name, 
                "numeric_val": val_as_float, 
                "display_val": raw_fil, # This is the "Y" in your entity:Y dict
                "original_row": row.copy() # Copy to avoid mutating original source
            })

        # 3. Filter/Sort Logic
        selected_matches = []
        if f_type == "top_n":
            sorted_pool = sorted(processed_pool, key=lambda x: x['numeric_val'], reverse=(sort_order == 'desc'))
            limit = min(max(int(f_val), 3), 10)
            selected_matches = sorted_pool[:limit]
        
        elif f_type == "numerical_range":
            low, high = float(f_val[0]), float(f_val[1])
            selected_matches = [d for d in processed_pool if low <= d['numeric_val'] <= high][:10]

        # 4. Construct Final Output Dict
        group_entities = {}
        group_data = []

        for match in selected_matches:
            # Add to entity:value dict
            group_entities[match['name']] = match['display_val']
            
            # Prepare row data: Remove the filter column
            pruned_row = match['original_row']
            if filter_key in pruned_row:
                del pruned_row[filter_key]
            
            group_data.append(pruned_row)

        return {
            "Group description": group_desc,
            "Group entities": group_entities,
            "Group data": group_data
        }

    except Exception as e:
        print(f"Extraction Error: {e}")
        return {}

In [55]:
extract_grouped_data(llm_output, final_data[0]['rows'])

{'Group description': 'Latest 7 chocolate manufacturers by founding date',
 'Group entities': {'Q sweet': 2019,
  'Fu Wan Chocolate': 2015,
  'Choco Togo': 2014,
  'Chocolala': 2013,
  'Castronovo Chocolate': 2012,
  'Malagos Chocolate': 2012,
  'Rick Jordan Chocolatier': 2011},
 'Group data': [{'Company': 'Q sweet',
   'Location': 'Taiwan',
   'Products': 'Chocolate truffles, bars, gift boxes',
   'Notes': ''},
  {'Company': 'Fu Wan Chocolate',
   'Location': 'Taiwan',
   'Products': 'Single-origin dark chocolates, Fermented and specialty bars combining cocoa with local flavors (e.g., lychee, tea, shrimp), Drinking chocolate, Limited editions based on Taiwanese terroir',
   'Notes': ''},
  {'Company': 'Choco Togo',
   'Location': 'Togo',
   'Products': 'Chocolate bars, hot chocolate, chocolate cake, and other chocolate products',
   'Notes': 'The first chocolate manufacturer in Togo.'},
  {'Company': 'Chocolala',
   'Location': 'Estonia',
   'Products': 'Single origin chocolate bars (

In [17]:
short = 0 #[0, 0, 0, 0, 0, 0, 0, 0]
long = 0
for l in [len(sample['rows']) for sample in final_data]:
    if l > 10:
        long += 1
    else:
        short += 1
        # short[l - 3] += 1

print(f"Short: {short}, Long: {long}")
print("=" * 100)

for i in range(3, 11):
    for t in final_data:
        if len(t['rows']) == i:
            print(t['table_name'])
            break

Short: 3942, Long: 3162
Comparison of OS emulation or virtualization apps on Android > Terminal emulation apps utilizing internal OS
ECHL awards > Team trophies
42nd Saturn Awards > Winners and nominees > Television > Programs
List of Renaissance band members > Members > Current
2011 ESPY Awards > Categories
List of Tennessee Titans head coaches > Key
List of Welsh flags > Regions, counties and cities > Traditional counties
ECHL awards > Individual trophies and awards


In [93]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import process_table_batches

# Assuming your tables are in a list of dicts
my_tables = random.sample(final_data, 10)

# Run the async loop
results = await process_table_batches(my_tables)

# Results will be a list of TableGroup objects/dicts
for my_tab, res in zip(my_tables, results):
    print(res.group_description)
    print(extract_grouped_data(res, my_tab['rows']))

Usage cost: 0.0032$ (Estimated: 0.00$): 100%|██████████████████████████████████████████| 10/10 [00:03<00:00,  3.30it/s]

The 5 sports with the highest total medal counts in mixed teams at the Olympics by Total medals.
{'Group description': 'The 5 sports with the highest total medal counts in mixed teams at the Olympics by Total medals.', 'Group entities': {'Total': 10, 'Tennis': 6, 'Tug of war': 1, 'Alpinism': 1, 'Polo': 1}, 'Group data': [{'Sport': 'Total', 'Gold': 4, 'Silver': 2, 'Bronze': 4}, {'Sport': 'Tennis', 'Gold': 1, 'Silver': 2, 'Bronze': 3}, {'Sport': 'Tug of war', 'Gold': 1, 'Silver': 0, 'Bronze': 0}, {'Sport': 'Alpinism', 'Gold': 1, 'Silver': 0, 'Bronze': 0}, {'Sport': 'Polo', 'Gold': 0, 'Silver': 0, 'Bronze': 1}]}
The 4 diplomatic missions of Indonesia in Oceania by host country name
{'Group description': 'The 4 diplomatic missions of Indonesia in Oceania by host country name', 'Group entities': {'Australia': 'Australia', 'Fiji': 'Fiji', 'New Zealand': 'New Zealand', 'Papua New Guinea': 'Papua New Guinea'}, 'Group data': [{'Host city': 'Canberra', 'Mission level': 'Embassy', 'Concurrent acc

In [94]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import extract_grouped_data

# Results will be a list of TableGroup objects/dicts
for my_tab, res in zip(my_tables, results):
    try:
        print(my_tab['table_name'])
        print(res.group_description)
        print(extract_grouped_data(res, my_tab['rows'])['Group entities'])
    except:
        pass

Mixed teams at the Olympics > Medal table > Medals by sport
The 5 sports with the highest total medal counts in mixed teams at the Olympics by Total medals.
{'Total': 10, 'Tennis': 6, 'Tug of war': 1, 'Alpinism': 1, 'Polo': 1}
List of diplomatic missions of Indonesia > Current missions > Oceania
The 4 diplomatic missions of Indonesia in Oceania by host country name
{'Australia': 'Australia', 'Fiji': 'Fiji', 'New Zealand': 'New Zealand', 'Papua New Guinea': 'Papua New Guinea'}
Largest known prime number > History
The 10 largest known prime numbers discovered by GIMPS by number of digits
Extraction Error: list index out of range
Comparison of mobile CRM systems > General
The 5 most recently updated mobile CRM systems by last updated year
Extraction Error: list index out of range
List of speech recognition software > Acoustic models and speech corpus (compilation)
The 5 open-source speech recognition applications by application name
{'CMU Sphinx': 'Yes', 'Julius': 'Yes', 'Kaldi': 'Yes', '

In [83]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import get_sampling_prompt

print(get_sampling_prompt(list(my_tables[0].values())[0], list(my_tables[0].values())[1]))

Analyze the Wikipedia table and identify a group of 3-10 entities.

    ### PRECISION RULES:
    1. THE GROUP NAME: Must be specific. Do NOT say 'Recent movies'. Say 'The 7 most recent movies' or 'Movies released 2010-2015'. 
    2. QUANTITY: Always include the count or the specific range bounds in the description.
    3. CONSTRAINTS: The result MUST contain at least 3 and no more than 10 rows.
    
    ### AVAILABLE STRATEGIES (Choose the best fit for this data):
    - [Numerical Strategy]: Threshold Filtering: Isolate 3-10 entities that fall above or below a specific significant threshold.
    - [Nested/List Strategy]: Complexity Count: Filter by the 'volume' of items in a cell (e.g., entities that have more than 5 but fewer than 10 associated countries/affiliates).
    - [Temporal Strategy]: Cutoff Logic: Filter for 3-10 entities that occurred specifically before or after a major milestone year.
    - [Categorical Strategy]: Multi-Column Pivot: Combine two categorical columns (e.g.,

In [112]:
import re

def smart_profile_table(table_rows):
    if not table_rows:
        return {}
    
    num_rows = len(table_rows)
    keys = list(table_rows[0].keys())
    profile = {"temporal": [], "numerical": [], "list": []}
    
    # Semantic keywords for temporal data
    temporal_keywords = ['year', 'date', 'founded', 'since', 'introduced', 'created', 'period', 'time', 'opened', 'released']
    
    for key in keys:
        col_name = str(key).lower()
        values = [str(row.get(key, "")).strip() for row in table_rows if row.get(key)]
        if not values: continue
        
        unique_count = len(set(values))
        
        # 1. SMART TEMPORAL CHECK
        # Must have a temporal keyword in the header OR match a strict year pattern
        has_temporal_name = any(word in col_name for word in temporal_keywords)
        # Check if values look like years/dates (e.g., 1995, 2024, or 12/05/2010)
        is_year_pattern = sum(1 for v in values if re.search(r'\b(1[789]\d{2}|20\d{2})\b', v)) / num_rows > 0.6
        
        if has_temporal_name or is_year_pattern:
            profile["temporal"].append({"col": key, "unique": unique_count})
            continue

        # 2. SMART NUMERICAL CHECK
        # Must be "mostly digits" for the majority of rows. 
        # We strip common symbols but expect a numeric core.
        numeric_count = 0
        for v in values:
            # Remove currency, commas, and units (km, mi, %)
            clean = re.sub(r'[^\d.]', '', v)
            if clean and clean.replace('.', '', 1).isdigit():
                numeric_count += 1
        
        if numeric_count / num_rows > 0.7:
            profile["numerical"].append({"col": key, "unique": unique_count})
            continue

        # 3. CATEGORICAL LIST CHECK
        # If it wasn't temporal or numerical, check if it's a repeating category.
        # Ratio check: Unique values should be between 3 and 80% of the total rows.
        if 3 <= unique_count <= (num_rows * 0.8):
            profile["list"].append({"col": key, "unique": unique_count})
            
    return profile

In [120]:
t = random.choice(my_tables)

smart_profile_table(t['rows'])

{'temporal': [{'col': 'Year', 'unique': 6}],
 'numerical': [{'col': 'Listicle', 'unique': 11}],
 'list': [{'col': 'Placement', 'unique': 7}]}

In [121]:
t

{'table_name': 'List of awards and nominations received by G-Dragon > Other accolades > Listicles',
 'rows': [{'Publisher': 'Arena Korea',
   'Year': 2008,
   'Listicle': 'Most Influential Men of the Year',
   'Placement': '1st'},
  {'Publisher': 'Business of Fashion',
   'Year': 2015,
   'Listicle': 'BoF 500',
   'Placement': 'Included'},
  {'Publisher': 'Complex',
   'Year': 2025,
   'Listicle': 'The 25 Best Dressed Celebs of the 21st Century',
   'Placement': '16th'},
  {'Publisher': 'Forbes',
   'Year': 2012,
   'Listicle': '2030 Power Leaders List',
   'Placement': 'Included'},
  {'Publisher': 'Gallup Korea',
   'Year': 2013,
   'Listicle': 'Top 10 Singers of the Year',
   'Placement': '10th'},
  {'Publisher': 'Golden Disc Awards',
   'Year': 2025,
   'Listicle': 'Golden Disc Powerhouse 40',
   'Placement': 'Included'},
  {'Publisher': 'GQ Korea',
   'Year': 2015,
   'Listicle': 'Men of the Year for 2015',
   'Placement': '1st'},
  {'Publisher': 'The Guardian',
   'Year': 2018,
  

In [5]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import process_small_tables_batch

small_table_sample = random.sample([t for t in final_data if len(t.get('rows', [])) <= 10], 5)
# await process_small_tables_batch(small_table_sample)

In [6]:
small_table_sample

[{'table_name': "List of teams and cyclists in the 2024 Giro d'Italia Women > Cyclists > By team",
  'rows': [{'No.': 41, 'Rider': 'Cédrine Kerbaol\xa0(FRA)', 'Pos.': 'DNF-4'},
   {'No.': 42, 'Rider': 'Alice Maria Arzuffi\xa0(ITA)', 'Pos.': 17},
   {'No.': 43, 'Rider': 'Laura Asencio\xa0(FRA)', 'Pos.': 31},
   {'No.': 44, 'Rider': 'Mylène de Zoete\xa0(NED)', 'Pos.': 99},
   {'No.': 45, 'Rider': 'Arianna Fidanza\xa0(ITA)', 'Pos.': 82},
   {'No.': 46, 'Rider': 'Marta Jaskulska\xa0(POL)', 'Pos.': 81},
   {'No.': 47, 'Rider': 'Kathrin Schweinberger\xa0(AUT)', 'Pos.': 92}]},
 {'table_name': "List of teams and cyclists in the 2024 Giro d'Italia Women > Cyclists > By team",
  'rows': [{'No.': 81, 'Rider': 'Pauliena Rooijakkers\xa0(NED)', 'Pos.': 4},
   {'No.': 82, 'Rider': 'Aniek van Alphen\xa0(NED)', 'Pos.': 76},
   {'No.': 83, 'Rider': 'Ceylin del Carmen Alvarado\xa0(NED)', 'Pos.': 58},
   {'No.': 84, 'Rider': 'Sanne Cant\xa0(BEL)', 'Pos.': 37},
   {'No.': 85, 'Rider': 'Millie Couzens\xa0(G

In [7]:
await process_small_tables_batch(small_table_sample[:3])

Usage cost: 0.0001$ (Estimated: 0.00$):  33%|██████████████▎                            | 1/3 [05:04<10:09, 304.88s/it]


CancelledError: 

In [23]:
for i in range(3, 11):
    for t in final_data:
        if len(t['rows']) == i:
            if 'List of countries' in t['table_name']:
                print(t['table_name'])
                print("="*100)
                break

List of countries that have the same national anthem > Countries that used to share the same melody
List of countries by diamond exports > Sources
List of countries without an airport > Sovereign states
List of countries that have gained independence from the United Kingdom > Evolution of dominions to independence
List of countries and dependencies and their capitals in native languages > F
List of countries and dependencies and their capitals in native languages > K
List of countries and dependencies and their capitals in native languages > E
List of countries by meat production > World production


In [25]:
for t in final_data:
    if len(t['rows']) <= 10:
        if 'List of countries' in t['table_name']:
            print(t['table_name'])

List of countries by diamond exports > Sources
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries that have gained independence from the United Kingdom > Evolution of dominions to independence
List of countries and dependencies and their capitals in native languages > D
List of countries and dependencies and their capitals in native languages > E
List of countries and dependencies and their capitals in native languages > F
List of countries and dependencies and their capitals in native languages > H
List of countries and dependencies and their capitals in native languages > I
List of countries and dependencies and their capitals in native languages > J
List of countries and dependencies a

In [26]:
import networkx as nx
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def clean_title(title):
    """Normalizes titles to increase clustering accuracy."""
    # Remove 'List of', 'Table of', punctuation, and extra whitespace
    t = title.lower()
    t = re.sub(r'\b(list of|table of|category:)\b', '', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return " ".join(t.split())

def cluster_and_deduplicate_tables(tables, similarity_threshold=0.85):
    """
    Groups tables by title similarity and returns one table per cluster.
    Input: List of dicts [{'table_name': '...', 'rows': [...]}, ...]
    """
    if not tables:
        return []

    # 1. Prepare Normalized Titles
    original_titles = [t.get('table_name', '') for t in tables]
    cleaned_titles = [clean_title(ttl) for ttl in original_titles]

    # 2. Vectorize using Character N-Grams (robust against typos/plurals)
    vectorizer = TfidfVectorizer(ngram_range=(2, 4), analyzer='char_wb')
    tfidf_matrix = vectorizer.fit_transform(cleaned_titles)

    # 3. Compute Sparse Similarity
    # Using linear_kernel or dot product is faster for normalized TF-IDF
    sim_matrix = cosine_similarity(tfidf_matrix)
    
    # 4. Build the Graph
    G = nx.Graph()
    G.add_nodes_from(range(len(tables)))
    
    # Efficiently find indices above threshold
    rows, cols = np.where(sim_matrix > similarity_threshold)
    
    # Add edges (r < c ensures we don't add self-loops or double edges)
    edges = [(r, c) for r, c in zip(rows, cols) if r < c]
    G.add_edges_from(edges)

    # 5. Extract Clusters and Select Representatives
    final_tables = []
    clusters = list(nx.connected_components(G))
    
    for cluster_indices in clusters:
        # Strategy: Pick the table with the most rows as the cluster representative
        # This ensures you keep the 'richest' data source for that topic
        best_index = max(cluster_indices, key=lambda i: len(tables[i].get('rows', [])))
        final_tables.append(tables[best_index])

    return final_tables

In [28]:
# Assuming 'all_tables' is your list of 30k dictionaries
deduped_dataset = cluster_and_deduplicate_tables(final_data, similarity_threshold=0.8)

print(f"Original: {len(final_data)} | Deduped: {len(deduped_dataset)}")

Original: 7104 | Deduped: 3378


In [29]:
for t in deduped_dataset:
    if len(t['rows']) <= 10:
        if 'List of countries' in t['table_name']:
            print(t['table_name'])

List of countries by diamond exports > Sources
List of countries that have gained independence from the United Kingdom > Evolution of dominions to independence
List of countries that have the same national anthem > Countries that share the same melody
List of countries without an airport > Sovereign states
List of countries without an airport > Non-self-governing territories
List of countries and territories by number of land borders > Artificial border crossings
List of countries that have gained independence from Portugal > Abandoned territories
List of countries by energy intensity > Sources


In [34]:
short = 0 #[0, 0, 0, 0, 0, 0, 0, 0]
midd = 0
long = 0
for l in [len(sample['rows']) for sample in deduped_dataset]:
    if l > 25:
        long += 1
    elif l > 12:
        midd += 1
    else:
        short += 1
        # short[l - 3] += 1

print(f"Short: {short}, Mid: {midd}, Long: {long}")
print("=" * 100)

for i in range(3, 13):
    for t in deduped_dataset:
        if len(t['rows']) == i:
            print(t['table_name'])
            break

Short: 1787, Mid: 799, Long: 792
Comparison of OS emulation or virtualization apps on Android > Terminal emulation apps utilizing internal OS
ECHL awards > Team trophies
51st Primetime Emmy Awards > Most major nominations
List of Welsh flags > Historical > Religious flags
2011 ESPY Awards > Categories
List of Tennessee Titans head coaches > Key
List of Welsh flags > Regions, counties and cities > Traditional counties
ECHL awards > Individual trophies and awards
List of motocross riders > World Championships by nationality
List of Indian Premier League awards > Best catch
